In [41]:
import sys

assert sys.version_info >= (3, 7)

In [42]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xarray as xr

In [43]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tf_keras

In [44]:
from packaging import version
import tensorflow as tf
from tensorflow.keras import layers, Sequential

assert version.parse(tf.__version__) >= version.parse("2.8.0")

In [45]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [46]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "nlp"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [47]:
if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. Neural nets can be very slow without a GPU.")
    if "google.colab" in sys.modules:
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")
    if "kaggle_secrets" in sys.modules:
        print("Go to Settings > Accelerator and select GPU.")

**READING THE DATASETS**

In [48]:
import xarray as xr

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

root_path = Path("/kaggle/input/stress-prediction-submit-prediction/")

/kaggle/input/stress-prediction-submit-prediction/train.csv
/kaggle/input/stress-prediction-submit-prediction/test.csv


**Reading Train Data**

In [49]:
# Get the train data and view it
text_train_raw_df = pd.read_csv(root_path / "train.csv", encoding='latin1') # encoding='latin1'
text_train_raw_df.head

/tmp/ipykernel_55/702576352.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  text_train_raw_df = pd.read_csv(root_path / "train.csv", encoding='latin1') # encoding='latin1'


<bound method NDFrame.head of                                             original_text  \
0       I went on a successful date with someone I fel...   
1       I was happy when my son got 90% marks in his e...   
2            I went to the gym this morning and did yoga.   
3       We had a serious talk with some friends of our...   
4       I went with grandchildren to butterfly display...   
...                                                   ...   
152494         The weight of wretchedness is crushing me.   
152495                          I feel empty in my bones.   
152496          Phone calls connect despite disconsolate.   
152497                    Friends see through to joyless.   
152498   My conscience yearns for relief from desolation.   

                                             cleaned_text is_modified  \
0       I went on a successful date with someone I fel...        True   
1       I was happy when my son got 90% marks in his e...        True   
2            I wen

In [50]:
# Shuffle rows
text_train_raw_df = text_train_raw_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(text_train_raw_df) # consider dropping white space in the sentences

                                            original_text  \
0              My family all went out to dinner together.   
1                    Phone calls connect despite choking.   
2                             The melancholy is too much.   
3                      My thoughts aches from depression.   
4                                                     NaN   
...                                                   ...   
152494                           wordless drains my will.   
152495  Are you lost in the fury, fearing that you'll ...   
152496                           Exercise battles sorrow.   
152497                   Friends see through to mournful.   
152498                This storm of washed out won't end.   

                                             cleaned_text is_modified  \
0              My family all went out to dinner together.        True   
1                    Phone calls connect despite choking.       False   
2                             The melancholy is 

**Reading and checking Test Data**

In [51]:
# Get the test data and view it

# Trying to open the test.csv file gave an error: UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb4 in position 4127: invalid start byte
# So I had to search for a way to check the encoding and use it to read the file.

import chardet

with open("/kaggle/input/stress-prediction-submit-prediction/test.csv", "rb") as f:
    result = chardet.detect(f.read())
print(result)


{'encoding': 'MacRoman', 'confidence': 0.6476582234668513, 'language': ''}


In [52]:
# reading test data
text_test_raw_df = pd.read_csv(root_path / "test.csv", encoding="MacRoman")
text_test_raw_df.head

<bound method NDFrame.head of         ID                                               text
0        1  Our national cricket team won the test match a...
1        2  I made a bunch of extra sales in my side busin...
2        3  I got a promotion today after 2 years of servi...
3        4  I woke up early in the morning at 7 am against...
4        5  when i logged on to account on m-turk i saw th...
...    ...                                                ...
1495  1496  I don't know why everyone is pretending to be ...
1496  1497  @IMKristenBell @lauralittlelegs pine nut.... C...
1497  1498  Never switched up, stayed down this the same p...
1498  1499  Do not grow weary of doing good' a lasting leg...
1499  1500     And with rich Fumes his sullen sences cheer'd.

[1500 rows x 2 columns]>

In [53]:
# Test data has no category. Only text as column
print(text_test_raw_df.columns.unique())

Index(['ID', 'text'], dtype='object')


In [54]:
# Drop the unnecessary columns, e.g. columns original_text, num_sentence ? 
text_test_df = text_test_raw_df.drop("ID", axis=1)
print(text_test_df)

                                                   text
0     Our national cricket team won the test match a...
1     I made a bunch of extra sales in my side busin...
2     I got a promotion today after 2 years of servi...
3     I woke up early in the morning at 7 am against...
4     when i logged on to account on m-turk i saw th...
...                                                 ...
1495  I don't know why everyone is pretending to be ...
1496  @IMKristenBell @lauralittlelegs pine nut.... C...
1497  Never switched up, stayed down this the same p...
1498  Do not grow weary of doing good' a lasting leg...
1499     And with rich Fumes his sullen sences cheer'd.

[1500 rows x 1 columns]


In [55]:
text_test_raw_df.isnull().sum()

ID      0
text    0
dtype: int64

**Checking Train data and cleaning it**

****Cleaning the Training data would include checking for null or Na values, looking at the skewness of the data and empty values ****

In [56]:
# Skewness of the data
print(text_train_raw_df['category'].value_counts())
# Sadness, Joy and affection have an overrepresentation of data rows. 
# This can affect the model training negatively. If I had the choice I would augment the data and add more rows of the other categories

category
sadness      45898
joy          36339
affection    34167
enjoyment    11143
bonding      10727
fear          2543
anger         1977
nature        1843
exercise      1202
love           159
surprise        66
Name: count, dtype: int64


**Drop all rows with nan or NAN**

In [57]:
# Get the Unique categories from the training set
# 12 unique categories were identified in the training set, i.e. 'affection' 'exercise' 'bonding' 'joy' 'enjoyment' 'nature' nan 'anger'
# 'fear' 'sadness' 'love' 'surprise'
no_na_text_train_df = text_train_raw_df.dropna()
print(no_na_text_train_df.category.unique())

['affection' 'sadness' 'joy' 'enjoyment' 'anger' 'nature' 'fear' 'bonding'
 'exercise' 'surprise' 'love']


In [58]:
# Checking for null values in data
no_na_text_train_df.isnull().sum()

original_text    0
cleaned_text     0
is_modified      0
num_sentence     0
category         0
dtype: int64

In [59]:
print(no_na_text_train_df.cleaned_text)

0                My family all went out to dinner together.
1                      Phone calls connect despite choking.
2                               The melancholy is too much.
3                        My thoughts aches from depression.
5         Something that made me happy recently is notic...
                                ...                        
152494                             wordless drains my will.
152495    Are you lost in the fury, fearing that you'll ...
152496                             Exercise battles sorrow.
152497                     Friends see through to mournful.
152498                  This storm of washed out won't end.
Name: cleaned_text, Length: 146064, dtype: object


In [60]:
# Checking for unique columns
print(no_na_text_train_df.columns.unique())

Index(['original_text', 'cleaned_text', 'is_modified', 'num_sentence',
       'category'],
      dtype='object')


In [61]:
print(no_na_text_train_df.cleaned_text)

0                My family all went out to dinner together.
1                      Phone calls connect despite choking.
2                               The melancholy is too much.
3                        My thoughts aches from depression.
5         Something that made me happy recently is notic...
                                ...                        
152494                             wordless drains my will.
152495    Are you lost in the fury, fearing that you'll ...
152496                             Exercise battles sorrow.
152497                     Friends see through to mournful.
152498                  This storm of washed out won't end.
Name: cleaned_text, Length: 146064, dtype: object


**Assign new column for str_cleaned_text to ensure it has only string**

In [62]:
# Assign new columns for str_cleaned_text as str to ensure there are no floats or ints in the column further below

str_cleaned_train_df = no_na_text_train_df.assign(str_cleaned_text=no_na_text_train_df["cleaned_text"].astype(str))

In [63]:
print(str_cleaned_train_df.str_cleaned_text) # new column str_cleaned_text has same number of rows as cleaned_text

0                My family all went out to dinner together.
1                      Phone calls connect despite choking.
2                               The melancholy is too much.
3                        My thoughts aches from depression.
5         Something that made me happy recently is notic...
                                ...                        
152494                             wordless drains my will.
152495    Are you lost in the fury, fearing that you'll ...
152496                             Exercise battles sorrow.
152497                     Friends see through to mournful.
152498                  This storm of washed out won't end.
Name: str_cleaned_text, Length: 146064, dtype: object


In [64]:
print(str_cleaned_train_df.category)

0         affection
1           sadness
2           sadness
3           sadness
5               joy
            ...    
152494      sadness
152495        anger
152496      sadness
152497      sadness
152498      sadness
Name: category, Length: 146064, dtype: object


In [65]:
str_cleaned_train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146064 entries, 0 to 152498
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   original_text     146064 non-null  object 
 1   cleaned_text      146064 non-null  object 
 2   is_modified       146064 non-null  object 
 3   num_sentence      146064 non-null  float64
 4   category          146064 non-null  object 
 5   str_cleaned_text  146064 non-null  object 
dtypes: float64(1), object(5)
memory usage: 7.8+ MB


In [66]:
str_cleaned_train_df.describe()

,num_sentence
count,146064.000000
mean,1.229393
std,1.084195
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,69.000000


**Cleaning Train dataframe of unnecessary columns**

In [67]:
# Drop the unnecessary columns, e.g. columns original_text
# drop all rows where column num_sentence > 2 because it will use too much computational power when encoding and running the Gemma model
# Decide whether you really want to run all 121999 rows of train_df in the model or just a sample. Same thing for the validation set. *********
str_cleaned_train_df.drop("original_text", axis=1)
str_cleaned_train_df.drop("is_modified", axis=1)
# text_train_df = text_train_raw_df.drop("num_sentence", axis=1) # drop this later
str_cleaned_train_df.head

<bound method NDFrame.head of                                             original_text  \
0              My family all went out to dinner together.   
1                    Phone calls connect despite choking.   
2                             The melancholy is too much.   
3                      My thoughts aches from depression.   
5       Something that made me happy recently is notic...   
...                                                   ...   
152494                           wordless drains my will.   
152495  Are you lost in the fury, fearing that you'll ...   
152496                           Exercise battles sorrow.   
152497                   Friends see through to mournful.   
152498                This storm of washed out won't end.   

                                             cleaned_text is_modified  \
0              My family all went out to dinner together.        True   
1                    Phone calls connect despite choking.       False   
2                 

In [68]:
# Select rows based on column values
text_train_df_more_sentences=str_cleaned_train_df[str_cleaned_train_df["num_sentence"] > 2.0] 
print("Selected rows where num_sentences is more than 2.0:\n", text_train_df_more_sentences)

Selected rows where num_sentences is more than 2.0:
                                             original_text  \
16      My husband's work has finally picked back up a...   
18      I was able to save $500 from my paycheck. I fe...   
41      i went to hobby lobby the other day and i saw ...   
64      The *sparkling* feeling you get after taking o...   
79      I mowed my lawn using my neighbor's mower and ...   
...                                                   ...   
152259  Every time you perform a selfless act, your br...   
152289  My boyfriend surprised me with tickets to Jers...   
152360  My son has difficulty with math and we have be...   
152404  My ungle son's marriage was happened in my nat...   
152414  Putting in new contact lenses.\n \n Watching g...   

                                             cleaned_text is_modified  \
16      My husband's work has finally picked back up a...        True   
18      I was able to save $500 from my paycheck. I fe...        Tru

In [69]:
for row in text_train_raw_df:
    str_cleaned_train_df.drop(str_cleaned_train_df[str_cleaned_train_df['num_sentence'] > 2.0].index, inplace= True)
        
# Select rows based on column values
text_train_df_more_sentences=str_cleaned_train_df[str_cleaned_train_df["num_sentence"] > 2.0] 
print("Selected rows where num_sentences is more than 2.0:\n", text_train_df_more_sentences)

Selected rows where num_sentences is more than 2.0:
 Empty DataFrame
Columns: [original_text, cleaned_text, is_modified, num_sentence, category, str_cleaned_text]
Index: []


In [70]:
# Train dataframe now has only num_sentences 2.0 or less.
str_cleaned_train_df.head

<bound method NDFrame.head of                                             original_text  \
0              My family all went out to dinner together.   
1                    Phone calls connect despite choking.   
2                             The melancholy is too much.   
3                      My thoughts aches from depression.   
5       Something that made me happy recently is notic...   
...                                                   ...   
152494                           wordless drains my will.   
152495  Are you lost in the fury, fearing that you'll ...   
152496                           Exercise battles sorrow.   
152497                   Friends see through to mournful.   
152498                This storm of washed out won't end.   

                                             cleaned_text is_modified  \
0              My family all went out to dinner together.        True   
1                    Phone calls connect despite choking.       False   
2                 

In [71]:
str_cleaned_train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 138918 entries, 0 to 152498
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   original_text     138918 non-null  object 
 1   cleaned_text      138918 non-null  object 
 2   is_modified       138918 non-null  object 
 3   num_sentence      138918 non-null  float64
 4   category          138918 non-null  object 
 5   str_cleaned_text  138918 non-null  object 
dtypes: float64(1), object(5)
memory usage: 7.4+ MB


In [72]:
str_cleaned_train_df.drop("original_text", axis=1)
#str_cleaned_train_df.drop("is_modified", axis=1)

str_cleaned_train_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 138918 entries, 0 to 152498
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   original_text     138918 non-null  object 
 1   cleaned_text      138918 non-null  object 
 2   is_modified       138918 non-null  object 
 3   num_sentence      138918 non-null  float64
 4   category          138918 non-null  object 
 5   str_cleaned_text  138918 non-null  object 
dtypes: float64(1), object(5)
memory usage: 7.4+ MB


In [73]:
# Keep the cleaned_text and category for categorization
str_cleaned_train_df.describe()

,num_sentence
count,138918.000000
mean,1.065765
std,0.247872
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,2.000000


In [74]:
# Split into 80%, 20% for training and validation
# Find index to where we want to split our data into our datasets
DataSplitRatios=(0.8,0.2) # Maybe one should check this sums up to 1?
n = len(str_cleaned_train_df)
split1ix = int(n*DataSplitRatios[0])
#split2ix = int(n*(DataSplitRatios[0]+DataSplitRatios[1]))
print(split1ix,n)

111134 138918


In [75]:
train_df = str_cleaned_train_df[:split1ix]
val_df = str_cleaned_train_df[split1ix:n]
#test_df = text_train_df[split2ix:n]
print(f"Total number of training rows: {len(train_df)}")
print(f"Total number of validation rows: {len(val_df)}")
#print(f"Total number of testing rows: {len(test_df)}")
print(f"Total number of rows: {len(train_df) + len(val_df)}")

Total number of training rows: 111134
Total number of validation rows: 27784
Total number of rows: 138918


In [76]:
print(str_cleaned_train_df['str_cleaned_text'].apply(type))


0         <class 'str'>
1         <class 'str'>
2         <class 'str'>
3         <class 'str'>
5         <class 'str'>
              ...      
152494    <class 'str'>
152495    <class 'str'>
152496    <class 'str'>
152497    <class 'str'>
152498    <class 'str'>
Name: str_cleaned_text, Length: 138918, dtype: object


In [77]:
# Checking the balance of the dataset
print(str_cleaned_train_df['category'].value_counts())

category
sadness      45898
joy          34688
affection    30918
enjoyment     9903
bonding       9881
fear          2543
anger         1977
nature        1720
exercise      1165
love           159
surprise        66
Name: count, dtype: int64


**Importing Gemma model through hugging face**

In [78]:
# Bring in the Gemma model
!pip install -U sentence-transformers git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview

  Cloning https://github.com/huggingface/transformers (to revision v4.56.0-Embedding-Gemma-preview) to /tmp/pip-req-build-j9lv1ict
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-j9lv1ict
  Running command git checkout -q 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Resolved https://github.com/huggingface/transformers to commit 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


**Logging into Hugging face**

In [79]:
# Login into Hugging Face Hub
from huggingface_hub import login
login() # Access Token: 

In [80]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/embeddinggemma-300M"
model = SentenceTransformer(model_id).to(device=device)

print(f"Device: {model.device}")
print(model)
print("Total number of parameters in the model:", sum([p.numel() for _, p in model.named_parameters()]))

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

Device: cuda:0
SentenceTransformer(
  (0): Transformer({'max_seq_length': 2048, 'do_lower_case': False, 'architecture': 'Gemma3TextModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 3072, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
  (3): Dense({'in_features': 3072, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
  (4): Normalize()
)
Total number of parameters in the model: 307581696


In [81]:
# Category labels on the training dataframe
category_labels = ['affection','exercise','bonding','joy','enjoyment','nature','anger','fear','sadness','love','surprise']#train_df.category.unique()
print(category_labels)

['affection', 'exercise', 'bonding', 'joy', 'enjoyment', 'nature', 'anger', 'fear', 'sadness', 'love', 'surprise']


In [82]:
# Cleaned text sentences from the training dataframe
# cleaned_text_sentences = train_df.cleaned_text
cleaned_text_sentences = str_cleaned_train_df.str_cleaned_text
print(cleaned_text_sentences)

0                My family all went out to dinner together.
1                      Phone calls connect despite choking.
2                               The melancholy is too much.
3                        My thoughts aches from depression.
5         Something that made me happy recently is notic...
                                ...                        
152494                             wordless drains my will.
152495    Are you lost in the fury, fearing that you'll ...
152496                             Exercise battles sorrow.
152497                     Friends see through to mournful.
152498                  This storm of washed out won't end.
Name: str_cleaned_text, Length: 138918, dtype: object


In [83]:

# sample cleaned_text sentences to run through model
sample_cleaned_df = str_cleaned_train_df.str_cleaned_text.iloc[100:2000]
print(len(sample_cleaned_df))

# Train data sentences to run through model
train_cleaned_df = train_df.str_cleaned_text
train_labels = train_df.category
print(train_labels)

# Validation data sentences
val_cleaned_df = val_df.str_cleaned_text
val_labels = val_df.category
print(val_labels)

1900
0         affection
1           sadness
2           sadness
3           sadness
5               joy
            ...    
122049      bonding
122051          joy
122052    affection
122053    enjoyment
122054          joy
Name: category, Length: 111134, dtype: object
122055          joy
122056         fear
122057      sadness
122058    affection
122059         fear
            ...    
152494      sadness
152495        anger
152496      sadness
152497      sadness
152498      sadness
Name: category, Length: 27784, dtype: object


In [84]:
# Changing uppcase sentences to lower case to enable correct embedding
def df_row_to_lower(sample_cleaned_df):
    index = 0
    changed_df = pd.DataFrame(sample_cleaned_df)
    for row in changed_df:
        str_sentence = str(row)
        if str_sentence[1:].isupper():
            changed_df.set_value(index, str_sentence.lower())
            changed_df.at[index, 'cleaned_text'] = str_sentence.lower()
            changed_df.replace(row,[str_sentence.lower()], inplace=True)
            index += 1
        changed_df = pd.DataFrame(sample_cleaned_df)
        return changed_df

all_lower_df = df_row_to_lower(sample_cleaned_df)

print(all_lower_df)


                                       str_cleaned_text
112   Ordering a Mushroom Pizza from Quatros. Easily...
113                            Hugs can't warm weeping.
114   Being hugged by a person that means a lot to m...
115   Another event that made me happy was finding o...
116                 pessimistic flows through my hands.
...                                                 ...
2198                         I BOUGHT A BIG TELEVISION.
2199                            Hope dims in cheerless.
2200                           My wife and I made love.
2201                                         AM IN WORK
2202                        Holidays remind of shadowy.

[1900 rows x 1 columns]


In [85]:
# placing the cleaned_text as elements in list

sample_list = []
def df_to_list(cleaned_text_df):
    sample_list = []
    for row in cleaned_text_df:
        sample_list.append(row)
    return sample_list

# Training data
Training_list = df_to_list(train_cleaned_df)
print(len(Training_list))

111134


In [86]:
# Calculate embeddings by calling model.encode()
label_embeddings = model.encode(category_labels, prompt_name="Classification")
text_embeddings = model.encode(Training_list, prompt_name="Classification")

# Calculate the embedding similarities
similarities = model.similarity(text_embeddings, label_embeddings)
print(similarities)

train_idx = similarities.argmax(1)
print(train_idx)

for text in Training_list[0:20]: # printing only 20 rows because 1900 is too many and makes it difficult to read everyything
  print(text, category_labels[train_idx[Training_list.index(text)]])

tensor([[0.3533, 0.3339, 0.3123,  ..., 0.2920, 0.3475, 0.2692],
        [0.4306, 0.4043, 0.4398,  ..., 0.4082, 0.4112, 0.4012],
        [0.4949, 0.4498, 0.4263,  ..., 0.7925, 0.5006, 0.4350],
        ...,
        [0.6082, 0.5250, 0.5470,  ..., 0.5211, 0.5999, 0.4677],
        [0.3014, 0.3646, 0.2929,  ..., 0.3968, 0.3000, 0.3397],
        [0.4136, 0.4256, 0.4167,  ..., 0.3232, 0.3905, 0.3472]])
tensor([5, 7, 8,  ..., 3, 8, 1])
My family all went out to dinner together. nature
Phone calls connect despite choking. fear
The melancholy is too much. sadness
My thoughts aches from depression. sadness
Something that made me happy recently is noticing that my strength has gone up in my lifts for weightlifting. I have been struggling to see gains in this department and am happy to finally see it happening. joy
Holidays remind of discouraged. sadness
Breathing reminds me of my depression. sadness
I ate some of my favorite meals (including bourbon steak) for dinner. nature
I sense discouraged thr

In [87]:
# category_labels = ['affection','exercise','bonding','joy','enjoyment','nature','anger','fear','sadness','love','surprise']
print(len(train_idx))
print(f" Predictions: {train_idx}, Labels: {train_labels}")
print(len(train_labels))

111134
 Predictions: tensor([5, 7, 8,  ..., 3, 8, 1]), Labels: 0         affection
1           sadness
2           sadness
3           sadness
5               joy
            ...    
122049      bonding
122051          joy
122052    affection
122053    enjoyment
122054          joy
Name: category, Length: 111134, dtype: object
111134


**Evaluation of Training**

In [88]:
# Creating a list of category_label indices # NB: this run takes VERY long. Hence using validation further below to make evaluation
y_true = pd.DataFrame(columns=['category'])
# Creating a list of category_label indices
categ_nr = []
for item in train_labels: # train_labels is a tensor while category_labels is a list
    for element in category_labels:
        if element == item:
            categ_nr.append(category_labels.index(element))

In [89]:
# Converting the list to a dataframe to use in the F1 matrix
y_true = pd.DataFrame(categ_nr, columns=['category'])
y_true.head

<bound method NDFrame.head of         category
0              0
1              8
2              8
3              8
4              3
...          ...
111129         2
111130         3
111131         0
111132         4
111133         3

[111134 rows x 1 columns]>

In [90]:
from sklearn.metrics import accuracy_score, f1_score  # Used AI here
from sklearn.metrics import fbeta_score

# Example: embeddings for N samples, label_embeddings for C classes
# embeddings: shape (N, D)
# label_embeddings: shape (C, D)
# y_true: ground truth labels (0..C-1)

# Compute cosine similarity
def cosine_similarity(a, b):
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return np.dot(a_norm, b_norm.T)

# Similarity matrix: (N, C)
sim_matrix = cosine_similarity(text_embeddings, label_embeddings)

# Predicted labels = index of max similarity
y_pred = np.argmax(sim_matrix, axis=1)

# Accuracy and F1
acc = accuracy_score(y_true, y_pred)
#f1 = f1_score(y_true, y_pred, average='micro') # Try average='micro', beta=0.5
f2 = fbeta_score(y_true, y_pred, average='micro', beta=0.5)
print(y_pred)
#print(f"Accuracy: {acc:.4f}")
#print(f"Weighted F1: {f1:.4f}")
print(f"Accuracy: {acc:.4f}")
print(f"Weighted Fbeta_score: {f2:.4f}")

[5 7 8 ... 3 8 1]
Accuracy: 0.4709
Weighted Fbeta_score: 0.4709


**Validation**

In [91]:
val_list = df_to_list(val_cleaned_df)
print(len(val_list))

27784


In [92]:
# Calculate embeddings by calling model.encode()
label_embeddings = model.encode(category_labels, prompt_name="Classification")
text_embeddings = model.encode(val_list, prompt_name="Classification")

# Calculate the embedding similarities
similarities = model.similarity(text_embeddings, label_embeddings)
print(similarities)

val_idx = similarities.argmax(1)
print(val_idx)
print(len(val_idx))

for text in val_list[0:20]: # printing only 20 rows because 1900 is too many and makes it difficult to read everyything
  print(text, category_labels[val_idx[val_list.index(text)]])

tensor([[0.3000, 0.3569, 0.2981,  ..., 0.3609, 0.2961, 0.2758],
        [0.4958, 0.4535, 0.4487,  ..., 0.3823, 0.4889, 0.4153],
        [0.4606, 0.4212, 0.4342,  ..., 0.4645, 0.4539, 0.4129],
        ...,
        [0.4762, 0.6469, 0.4458,  ..., 0.6716, 0.4630, 0.4055],
        [0.5078, 0.4501, 0.4817,  ..., 0.6865, 0.4806, 0.4659],
        [0.4188, 0.4280, 0.4275,  ..., 0.5127, 0.4261, 0.4235]])
tensor([3, 7, 6,  ..., 8, 8, 8])
27784
I'm happy that my Monday is over and I got thru one work day of the week. joy
ï¿½ï¿½ï¿½ My courage always rises at every attempt to intimidate me.ï¿½ï¿½ï¿½\n-Elizabeth Bennett (Pride and Prejudice)\n#Quotes #Courage #FaceYourFears fear
This storm of strangled won't end. anger
My daughter is reportedly the most developed artist that her teacher has seen in her teaching career. nature
Time for some despair #SDR3 #despair #fuckthisanime sadness
My hands pulses with dim. affection
I got a lot of my work assignments done at work. It really saved me through a lot

**Evaluating the validation set**

In [93]:
print(len(val_idx)) # the val_idx i.e. predictions are 27784

27784


In [94]:
# Creating a list of category_label indices
categ_nr = []
for item in val_labels: # val_labels is a tensor while category_labels is a list
    for element in category_labels:
        if element == item:
            categ_nr.append(category_labels.index(element))

In [95]:
print(categ_nr[:10])
print(len(categ_nr))

[3, 7, 8, 0, 7, 8, 3, 8, 0, 0]
27784


In [96]:
# Converting the list to a dataframe to use in the F1 matrix
val_labels_true = pd.DataFrame(categ_nr, columns=['category'])
val_labels_true.head

<bound method NDFrame.head of        category
0             3
1             7
2             8
3             0
4             7
...         ...
27779         8
27780         6
27781         8
27782         8
27783         8

[27784 rows x 1 columns]>

In [97]:
from sklearn.metrics import accuracy_score, f1_score  # Used AI here
from sklearn.metrics import fbeta_score
# Example: embeddings for N samples, label_embeddings for C classes
# embeddings: shape (N, D)
# label_embeddings: shape (C, D)
# y_true: ground truth labels (0..C-1)

# Compute cosine similarity
def cosine_similarity(a, b):
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return np.dot(a_norm, b_norm.T)

# Similarity matrix: (N, C)
sim_matrix = cosine_similarity(text_embeddings, label_embeddings)

# Predicted labels = index of max similarity
y_pred = np.argmax(sim_matrix, axis=1)

# Accuracy and F1
acc = accuracy_score(val_labels_true, y_pred)
#f1 = f1_score(val_labels_true, y_pred, average="weighted")
f2 = fbeta_score(val_labels_true, y_pred, average='micro', beta=0.5)
print(y_pred)
print(f"Accuracy: {acc:.4f}")
print(f"Weighted Fbeta_score: {f2:.4f}")

[3 7 6 ... 8 8 8]
Accuracy: 0.4692
Weighted Fbeta_score: 0.4692


### It appears that the accuracy and F1 calculated are either inaccurate or underperforming. But from a human perspective when I look at the similarity results they look correctly attributed to each text. I will need to try another metric or model.

**Testing the test dataset**

In [98]:
text_test_raw_df.iloc[0:500]

,ID,text
0,1,Our national cricket team won the test match a...
1,2,I made a bunch of extra sales in my side busin...
2,3,I got a promotion today after 2 years of servi...
3,4,I woke up early in the morning at 7 am against...
4,5,when i logged on to account on m-turk i saw th...
...,...,...
495,496,@yesnicksearcy @counterparty Although I don't ...
496,497,You can tell the camp isn't happy purely throu...
497,498,Migraine hangover all day. Stood up to do dish...
498,499,Migraine hangover all day. Stood up to do dish...


In [99]:
copy_test_df = text_test_raw_df.copy()
copy_test_df.drop("ID", axis=1)

,text
0,Our national cricket team won the test match a...
1,I made a bunch of extra sales in my side busin...
2,I got a promotion today after 2 years of servi...
3,I woke up early in the morning at 7 am against...
4,when i logged on to account on m-turk i saw th...
...,...
1495,I don't know why everyone is pretending to be ...
1496,@IMKristenBell @lauralittlelegs pine nut.... C...
1497,"Never switched up, stayed down this the same p..."
1498,Do not grow weary of doing good' a lasting leg...


In [100]:
copy_test_df.head

<bound method NDFrame.head of         ID                                               text
0        1  Our national cricket team won the test match a...
1        2  I made a bunch of extra sales in my side busin...
2        3  I got a promotion today after 2 years of servi...
3        4  I woke up early in the morning at 7 am against...
4        5  when i logged on to account on m-turk i saw th...
...    ...                                                ...
1495  1496  I don't know why everyone is pretending to be ...
1496  1497  @IMKristenBell @lauralittlelegs pine nut.... C...
1497  1498  Never switched up, stayed down this the same p...
1498  1499  Do not grow weary of doing good' a lasting leg...
1499  1500     And with rich Fumes his sullen sences cheer'd.

[1500 rows x 2 columns]>

In [101]:
sample_list = []
def test_df_to_list(test_df):
    sample_list = []
    for row in test_df:
        sample_list.append(row)
    return sample_list

In [102]:
sample_test_df = copy_test_df.text[:]

In [103]:
sampled_test_list = test_df_to_list(sample_test_df)
print(len(sampled_test_list))

1500


In [104]:
text_test_embeddings = model.encode(sampled_test_list, prompt_name="Classification")

# Calculate the embedding similarities
similarities = model.similarity(text_test_embeddings, label_embeddings)
print(similarities)

idx = similarities.argmax(1)
print(idx)

for text in sampled_test_list[0:10]: # printing only 20 rows because 1900 is too many and makes it difficult to read everyything
  print(text, category_labels[idx[sampled_test_list.index(text)]])

tensor([[0.1975, 0.2623, 0.2267,  ..., 0.2181, 0.1979, 0.2186],
        [0.2499, 0.3110, 0.2591,  ..., 0.2204, 0.2349, 0.3087],
        [0.3154, 0.3516, 0.3127,  ..., 0.3066, 0.3055, 0.3428],
        ...,
        [0.4653, 0.4016, 0.4476,  ..., 0.4447, 0.4816, 0.3658],
        [0.4491, 0.4376, 0.4469,  ..., 0.3957, 0.4517, 0.3835],
        [0.4084, 0.4016, 0.3762,  ..., 0.3697, 0.3797, 0.3789]])
tensor([3, 3, 3,  ..., 9, 9, 6])
Our national cricket team won the test match and the series this morning. This win sealed our position as the no.1 Test team in the world. This made me very happy today. joy
I made a bunch of extra sales in my side business last night. joy
I got a promotion today after 2 years of service at my job. joy
I woke up early in the morning at 7 am against daily of 8:30 am surprise
when i logged on to account on m-turk i saw that all the pending hits have got approved and i still had 100% acceptance surprise
I slow cooked a chuck roast on my stove that came out great. na

### Analysis: The results of the similarity categorization appear at first glance to look correct. But because we have no labels in the test set, it is difficult to evaluate it.

In [105]:
# Set up hyperparameters. 
# Decide on metrics and loss_functions
# auto-tune the hyperparameters
# Set up a visualization of training rounds and do cross_validations
# Tensor_board?
# early -stopping

****SEQUENTIAL MODEL ********

In [106]:
# 1. Training data set
str_cleaned_train_df

,original_text,cleaned_text,is_modified,num_sentence,category,str_cleaned_text
0,My family all went out to dinner together.,My family all went out to dinner together.,True,1.0,affection,My family all went out to dinner together.
1,Phone calls connect despite choking.,Phone calls connect despite choking.,False,1.0,sadness,Phone calls connect despite choking.
2,The melancholy is too much.,The melancholy is too much.,False,1.0,sadness,The melancholy is too much.
3,My thoughts aches from depression.,My thoughts aches from depression.,False,1.0,sadness,My thoughts aches from depression.
5,Something that made me happy recently is notic...,Something that made me happy recently is notic...,True,2.0,joy,Something that made me happy recently is notic...
...,...,...,...,...,...,...
152494,wordless drains my will.,wordless drains my will.,False,1.0,sadness,wordless drains my will.
152495,"Are you lost in the fury, fearing that you'll ...","Are you lost in the fury, fearing that you'll ...",False,1.0,anger,"Are you lost in the fury, fearing that you'll ..."
152496,Exercise battles sorrow.,Exercise battles sorrow.,False,1.0,sadness,Exercise battles sorrow.
152497,Friends see through to mournful.,Friends see through to mournful.,False,1.0,sadness,Friends see through to mournful.


In [107]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import time

In [108]:
# 2. Encode Labels
label_encoder = LabelEncoder()
str_cleaned_train_df['label'] = label_encoder.fit_transform(str_cleaned_train_df['category'])
num_classes = len(label_encoder.classes_)
print(num_classes)
y = to_categorical(str_cleaned_train_df['label'], num_classes=num_classes)
print(y)
print(len(y))

11
[[1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 ...
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 1. 0.]]
138918


In [109]:
# 3. Text Vectorization Layer
# -----------------------------
max_tokens = 1000   # Vocabulary size
output_sequence_length = 200  # Max sequence length

vectorizer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=output_sequence_length
)

# Adapt vectorizer to training text
vectorizer.adapt(str_cleaned_train_df['cleaned_text'])


I0000 00:00:1768737600.885506      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13379 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [110]:
# 4. Tokenize and pad sequences
# -----------------------------
MAX_WORDS = 5000   # Vocabulary size
MAX_LEN = 10       # Max sequence length

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(str_cleaned_train_df['cleaned_text'])
sequences = tokenizer.texts_to_sequences(str_cleaned_train_df['cleaned_text'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')

In [111]:
print(X)

[[  3  53  51 ... 155   0   0]
 [150 733 726 ...   0   0   0]
 [  7 358  17 ...   0   0   0]
 ...
 [569 903 260 ...   0   0   0]
 [ 49  76  81 ...   0   0   0]
 [ 30 636  10 ...   0   0   0]]


In [112]:
# 5. Train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42  # X is the raw dataset including train, val, test, y is the encoded labels to categorical
)

print(X_test)

[[   2  281   53 ...    0    0    0]
 [  12    4   37 ...   11    4  107]
 [2175    8   73 ... 1043 1405    1]
 ...
 [   2  152  139 ...   62 1897  465]
 [ 513    3    1 ...   30  138  188]
 [  13  135 3211 ...  295   82   52]]


In [117]:
# 6. Build Sequential model, fbeta micro score, model compile
# 
# -----------------------------


fbeta_micro = tf.keras.metrics.FBetaScore(
    average='micro',
    beta=1.0,
    threshold=None,
    name='fbeta_score',
    dtype=None
)

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')  # Multi-class output
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy', fbeta_micro]
)

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 10, 64)            320000    
                                                                 
 global_average_pooling1d_1  (None, 64)                0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dense_2 (Dense)             (None, 64)                4160      
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_3 (Dense)             (None, 11)                715       
                                                                 
Total params: 324875 (1.24 MB)
Trainable params: 324875 (1.24 MB)
Non-trainable params: 0 (0.00 Byte)
__________________

In [118]:
# 7. Train the model
# -----------------------------
callbacks = tf.keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=3,restore_best_weights = True)

start = time.perf_counter()
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=4,
    verbose=1,
    callbacks=callbacks
)
end = time.perf_counter()
print(f"Time to run: {end - start:.2f}s",)

Epoch 1/10


I0000 00:00:1768738210.113335     420 service.cc:152] XLA service 0x7a0214a53660 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768738210.113380     420 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1768738210.196885     420 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1768738210.385812     420 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


27784/27784 [==============================] - 106s 4ms/step - loss: 0.5533 - accuracy: 0.8165 - fbeta_score: 0.8165 - val_loss: 0.4366 - val_accuracy: 0.8503 - val_fbeta_score: 0.8503
Epoch 2/10
27784/27784 [==============================] - 97s 3ms/step - loss: 0.4111 - accuracy: 0.8634 - fbeta_score: 0.8634 - val_loss: 0.4239 - val_accuracy: 0.8564 - val_fbeta_score: 0.8564
Epoch 3/10
27784/27784 [==============================] - 98s 4ms/step - loss: 0.3783 - accuracy: 0.8711 - fbeta_score: 0.8711 - val_loss: 0.4264 - val_accuracy: 0.8586 - val_fbeta_score: 0.8586
Epoch 4/10
27784/27784 [==============================] - 97s 4ms/step - loss: 0.3602 - accuracy: 0.8783 - fbeta_score: 0.8783 - val_loss: 0.4234 - val_accuracy: 0.8592 - val_fbeta_score: 0.8592
Epoch 5/10
27784/27784 [==============================] - 97s 3ms/step - loss: 0.3460 - accuracy: 0.8837 - fbeta_score: 0.8837 - val_loss: 0.4318 - val_accuracy: 0.8582 - val_fbeta_score: 0.8582
Epoch 6/10
27784/27784 [===========

In [123]:
# 8. Evaluate
# -----------------------------
#loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
#print(f"Test Accuracy: {accuracy:.2f}")

loss, accuracy, fbeta_score = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.2f}, FBeta_Score: {fbeta_score}")

Test Accuracy: 0.86, FBeta_Score: 0.8592354655265808


In [128]:
# 8. Predict on new text
seqs = tokenizer.texts_to_sequences(text_test_df['text'])
padded = pad_sequences(seqs, maxlen=MAX_LEN, padding='post')
predictions = model.predict(padded)
preds_list = []
for text, pred in zip(text_test_df['text'], predictions):
    predicted_label = label_encoder.inverse_transform([np.argmax(pred)])[0]
    preds_list.append(predicted_label)
    #print(f"'{text}' → {predicted_label}")
print(preds_list[0:100])  # Show a few of the predictions for the test data
print(len(preds_list)) # number of predictions is equal to the number of text_test_df rows

47/47 [==============================] - 0s 1ms/step
['joy', 'joy', 'joy', 'joy', 'joy', 'joy', 'joy', 'joy', 'joy', 'joy', 'joy', 'joy', 'joy', 'joy', 'joy', 'anger', 'anger', 'anger', 'joy', 'anger', 'anger', 'affection', 'anger', 'anger', 'anger', 'anger', 'anger', 'anger', 'affection', 'anger', 'sadness', 'sadness', 'sadness', 'sadness', 'sadness', 'sadness', 'affection', 'sadness', 'sadness', 'anger', 'sadness', 'sadness', 'affection', 'sadness', 'sadness', 'sadness', 'sadness', 'sadness', 'fear', 'sadness', 'affection', 'affection', 'affection', 'joy', 'affection', 'bonding', 'joy', 'joy', 'joy', 'bonding', 'joy', 'joy', 'joy', 'exercise', 'affection', 'joy', 'affection', 'bonding', 'affection', 'joy', 'affection', 'affection', 'bonding', 'affection', 'affection', 'joy', 'joy', 'affection', 'affection', 'joy', 'affection', 'joy', 'joy', 'joy', 'bonding', 'joy', 'enjoyment', 'bonding', 'affection', 'joy', 'affection', 'bonding', 'joy', 'joy', 'affection', 'joy', 'affection', 'enjo

**ANALYSIS ABOUT HYPERPARAMETERS**
I considered runing a tuner with several hyperparameters in a Sequential model with several Conv2D layers, a max_pooling layer and auto-tune hyperparameters,kernel initialization and batchnormalisation in order to push the accuracy upto and beyond 90%. However, I felt satisfied with what the simple sequential model had accomplished. I wanted to concentrate on some literature analysis to integrate into the project report.

**Loss_Functions**
I would have liked to include a Confusion matrix in order to see which categories the model was confusing the sentences with. Fscore average=weighted is said to be good for labels that are not balanced, i.e. if there are more labels of some categories than others (see page 126 in Géron) In this competition, we are asked to calculate the Fbeta score micro. Fbeta micro calculates the FI score based on TP, FP, FN of each category, and is recommended when categories are imbalanced as we have in this case (sadness, joy and affection are the largest categories in the dataset category: sadness  45898, joy   34688, affection  30918). Fbeta micro gives a better representation of underrepresented classes.

**Analysis on the models embeddinggemma-300M and Sequential**
The embeddinggemma-300M gave an accuracy of 0.47 and fbeta score of 0.47, which was a bit surprising and disappointing. I had hoped for more. The sequential model has a globalpooling layer which handles inputs of variable lengths. It has a dense layer with activation ReLU allows the model to learn complex feature combinations which the different sentences provide. The question is if more dense layers would have improved the learning. The dropout aimed to prevent overfitting on the model on the data.

**AI USE**
I consulted AI for the Fbeta_score but also TensorFlows documentation page https://www.tensorflow.org/api_docs/python/tf/keras/metrics/FBetaScore, and a simple tokenization of the text (although I could have taken the tf tokenizer), as well as the simple Sequential model.

In [ ]:
# VG: Evaluation compare with literature on similar/different models on the same problem

Literature Review
There are other algorithms have been used to predict stress. The data includes not just text, but also snoring rate, respiration rate, heart rate, and other factors.  The Random Forest Classifier has been used on these data sets(1). In another study the authors proposed a semi-supervised machine learning model (DASentimental) to extract depression, anxiety and stress from written text (2). In this study the authors used a multilayer perceptron neural network trained on word sequences and semantic network distances can achieve state-of-art, cross-validated predictions for depression (R = 0.7), anxiety (R = 0.44) and stress (R = 0.52). Indeed they conclude that suicidal tendencies may be deduced by looking at more complex patterns between feelings that individuals recall having recently and depression, anxiety and stress (DAS). The DASentimental study ran three kinds of models: (i) decision tree, (ii) multi-layer perceptron (MLP), and (iii) recurrent neural network (Long-Short Term Memory or LSTM). The LSTM had two layers with four feature cells in each layer and a drop-out rate of 20%. The MLP had 2 hidden layers with 25 neurons each and a drop-out layer of 20%.
In another study a pre-trained BERT model with CNN classifier is used and is reported to achieve high performance results on AG News and Amazon product reviews text datasets (3). The same article goes on to mention that Random Forest Classifier achieved 97% accuracy. The overall conclusion was that BERT models appeared to attain best results but Random forest and XGBoost (Extreme Gradient Boosting) are also powerful machine learning algorithms. 
My conclusion is that these models outperform my simple Sequential Model by a wide margin. My model gave an accuracy of 86% whilst these models have an accuracy way over 90%.

 1 Stress_detection_using_ai_and_machine_learning. 
 2 DASentimental: Detecting depression, anxiety and stress in texts via emotional recall, cognitive networks and machine learning. 2110.13710
3 Text-Based Stress Detection And Classification Using Machine Learning. IJRTI
